In [1]:
import typing as t
import math


def compute_max_sequences_rhm(
    num_classes: int,
    vocab_size: int,  # This is 'v' in the original code
    num_layers: int,  # This is 'L' 
    tuple_size: int   # This is 's'
) -> int:
    """Compute maximum sequences for Random Hierarchy Model.
    
    Based on the formula: max_data = n * m^((s^L - 1) / (s - 1))
    Where m (num_synonyms) appears to be derived from vocab_size in some way.
    
    Looking at the original code, m seems to be a parameter for multiplicity,
    but from your config it looks like you might be using a different interpretation.
    """
    # From the original sample_rules function, it seems like:
    # - Level 0: n * m rules (where n = num_classes)  
    # - Level i>0: v * m rules (where v = vocab_size)
    
    # The exponent calculation
    if tuple_size == 1:
        exponent = num_layers
    else:
        exponent = (tuple_size ** num_layers - 1) // (tuple_size - 1)
    
    # Based on the code structure, it seems m might equal vocab_size
    # or there might be a different interpretation
    # Let me calculate for different assumptions:
    return exponent


def analyze_config_list() -> None:
    """Analyze the specific configuration list provided."""
    
    config_list = [
        (2, 2),  # (num_layers, tuple_size) - Shallow, low multiplicity  
        (3, 2),  # Medium depth, low multiplicity
        (2, 4),  # Shallow, high multiplicity
        (4, 2),  # Deep, low multiplicity
        (3, 3),  # Medium depth, medium multiplicity
    ]
    
    # Your settings
    vocab_size = 32
    num_classes = 10
    samples_per_config = 1000
    
    print("=== Configuration Analysis ===")
    print(f"vocab_size = {vocab_size}")
    print(f"num_classes = {num_classes}")
    print(f"samples_per_config = {samples_per_config}")
    print()
    
    print("Config | L | s | Exponent | Max Sequences (m=vocab_size=32)")
    print("-" * 65)
    
    for i, (num_layers, tuple_size) in enumerate(config_list):
        exponent = compute_max_sequences_rhm(num_classes, vocab_size, num_layers, tuple_size)
        
        # Assuming m = vocab_size = 32 (this is a common interpretation)
        max_sequences = num_classes * (vocab_size ** exponent)
        
        description = [
            "Shallow, low multiplicity",
            "Medium depth, low multiplicity", 
            "Shallow, high multiplicity",
            "Deep, low multiplicity",
            "Medium depth, medium multiplicity"
        ][i]
        
        print(f"{i+1:6d} | {num_layers} | {tuple_size} | {exponent:8d} | {max_sequences:15,d}")
        print(f"       {description}")
        print()


def detailed_breakdown() -> None:
    """Provide detailed breakdown of the calculations."""
    
    config_list = [
        (2, 2),  # Shallow, low multiplicity
        (3, 2),  # Medium depth, low multiplicity
        (2, 4),  # Shallow, high multiplicity
        (4, 2),  # Deep, low multiplicity
        (3, 3),  # Medium depth, medium multiplicity
    ]
    
    vocab_size = 32
    num_classes = 10
    
    print("=== Detailed Breakdown ===")
    print("Formula: max_sequences = num_classes * vocab_size^exponent")
    print("Where exponent = (tuple_size^num_layers - 1) / (tuple_size - 1)")
    print()
    
    for i, (L, s) in enumerate(config_list):
        print(f"Configuration {i+1}: L={L}, s={s}")
        
        if s == 1:
            exponent = L
            print(f"  Special case (s=1): exponent = L = {L}")
        else:
            numerator = s ** L - 1
            denominator = s - 1
            exponent = numerator // denominator
            print(f"  Exponent = ({s}^{L} - 1) / ({s} - 1) = ({numerator}) / ({denominator}) = {exponent}")
        
        # Geometric series interpretation
        terms = [s ** i for i in range(L)]
        print(f"  Geometric series: {' + '.join([f'{s}^{i}' for i in range(L)])} = {' + '.join(map(str, terms))} = {sum(terms)}")
        
        max_sequences = num_classes * (vocab_size ** exponent)
        print(f"  Max sequences = {num_classes} * {vocab_size}^{exponent} = {max_sequences:,}")
        
        # Scientific notation for very large numbers
        if max_sequences > 1e6:
            print(f"  Scientific notation: {max_sequences:.2e}")
        
        print()


def compare_with_sample_size() -> None:
    """Compare theoretical max with actual sample size."""
    
    config_list = [
        (2, 2),  # Shallow, low multiplicity
        (3, 2),  # Medium depth, low multiplicity
        (2, 4),  # Shallow, high multiplicity
        (4, 2),  # Deep, low multiplicity
        (3, 3),  # Medium depth, medium multiplicity
    ]
    
    vocab_size = 32
    num_classes = 10
    samples_per_config = 1000
    
    print("=== Sample Size vs Theoretical Maximum ===")
    print("Config | Max Sequences | Sample Size | Coverage %")
    print("-" * 55)
    
    total_theoretical = 0
    total_samples = len(config_list) * samples_per_config
    
    for i, (L, s) in enumerate(config_list):
        exponent = (s ** L - 1) // (s - 1) if s > 1 else L
        max_sequences = num_classes * (vocab_size ** exponent)
        coverage = (samples_per_config / max_sequences) * 100 if max_sequences > 0 else 0
        
        total_theoretical += max_sequences
        
        print(f"{i+1:6d} | {max_sequences:13,d} | {samples_per_config:11,d} | {coverage:8.6f}%")
    
    print("-" * 55)
    print(f"Total  | {total_theoretical:13,d} | {total_samples:11,d} | {(total_samples/total_theoretical)*100:8.6f}%")


def memory_usage_estimate() -> None:
    """Estimate memory usage for the datasets."""
    
    config_list = [
        (2, 2), (3, 2), (2, 4), (4, 2), (3, 3)
    ]
    
    vocab_size = 32
    num_classes = 10
    samples_per_config = 1000
    
    print("=== Memory Usage Estimates ===")
    print("Assuming float32 features and int64 labels")
    print()
    
    total_memory = 0
    
    for i, (L, s) in enumerate(config_list):
        exponent = (s ** L - 1) // (s - 1) if s > 1 else L
        feature_size = s ** L  # Final feature vector size
        
        # Memory per sample (rough estimate)
        bytes_per_feature = 4  # float32
        bytes_per_label = 8    # int64
        memory_per_sample = feature_size * bytes_per_feature + bytes_per_label
        total_config_memory = samples_per_config * memory_per_sample
        
        total_memory += total_config_memory
        
        print(f"Config {i+1} (L={L}, s={s}):")
        print(f"  Feature vector size: {feature_size:,}")
        print(f"  Memory per sample: {memory_per_sample:,} bytes ({memory_per_sample/1024:.1f} KB)")
        print(f"  Total for {samples_per_config:,} samples: {total_config_memory/1024/1024:.1f} MB")
        print()
    
    print(f"Total estimated memory: {total_memory/1024/1024:.1f} MB ({total_memory/1024/1024/1024:.2f} GB)")


def test_calculations() -> None:
    """Test the calculations with your specific parameters."""
    print("=== Testing Calculations ===")
    
    # Run all analyses
    analyze_config_list()
    print("\n" + "="*60 + "\n")
    
    detailed_breakdown()
    print("\n" + "="*60 + "\n")
    
    compare_with_sample_size()
    print("\n" + "="*60 + "\n")
    
    memory_usage_estimate()


if __name__ == "__main__":
    test_calculations()

=== Testing Calculations ===
=== Configuration Analysis ===
vocab_size = 32
num_classes = 10
samples_per_config = 1000

Config | L | s | Exponent | Max Sequences (m=vocab_size=32)
-----------------------------------------------------------------
     1 | 2 | 2 |        3 |         327,680
       Shallow, low multiplicity

     2 | 3 | 2 |        7 | 343,597,383,680
       Medium depth, low multiplicity

     3 | 2 | 4 |        5 |     335,544,320
       Shallow, high multiplicity

     4 | 4 | 2 |       15 | 377,789,318,629,571,617,095,680
       Deep, low multiplicity

     5 | 3 | 3 |       13 | 368,934,881,474,191,032,320
       Medium depth, medium multiplicity



=== Detailed Breakdown ===
Formula: max_sequences = num_classes * vocab_size^exponent
Where exponent = (tuple_size^num_layers - 1) / (tuple_size - 1)

Configuration 1: L=2, s=2
  Exponent = (2^2 - 1) / (2 - 1) = (3) / (1) = 3
  Geometric series: 2^0 + 2^1 = 1 + 2 = 3
  Max sequences = 10 * 32^3 = 327,680

Configuration 2: